# Phase 5: Hyperparameter Sensitivity & Ablation Studies

## Objective
Understand which hyperparameters have the biggest impact on Mountain Car learning. Identify critical thresholds and interactions.

## Key Question
**Which hyperparameters matter most?** Are there specific regions of parameter space where performance breaks down?

## Experimental Design
- **Method**: Use best tabular method from Phase 4
- **One-factor-at-a-time**: Vary one hyperparameter while keeping others fixed
- **Parameters studied**:
  - Learning rate (α): {0.01, 0.05, 0.1, 0.2, 0.5}
  - Discount factor (γ): {0.5, 0.75, 0.9, 0.95, 0.99}
  - Exploration decay (ε_decay): {0.98, 0.99, 0.995, 0.999}
  - Min epsilon (ε_min): {0.01, 0.05, 0.1}
- **Robustness**: 3 seeds per configuration
- **Metrics**: Final performance, convergence speed, stability

## Setup & Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
project_root = Path(".").resolve().parent
sys.path.insert(0, str(project_root / "src"))

from environment_utils import create_env, StateDiscretizer
from evaluation import train_agent_legacy as train_agent
from plotting import plot_training_curve, plot_success_curve
from agents.tabular_agents import QLearning

print("✓ Imports successful")

## Configuration

In [ ]:
# Best settings from Phase 4
BEST_ALPHA = 0.1
BEST_GAMMA = 0.99
BEST_EPSILON_DECAY = 0.995
BEST_EPSILON_MIN = 0.01

# Discretization from Phase 2
DISCRETIZATION_BINS = 20

# Experiment config
N_EPISODES = 500
N_SEEDS = 3
ENVIRONMENT = "min_steps"

# Parameters to sweep
ALPHA_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5]
GAMMA_VALUES = [0.5, 0.75, 0.9, 0.95, 0.99]
EPSILON_DECAY_VALUES = [0.98, 0.99, 0.995, 0.999]
EPSILON_MIN_VALUES = [0.01, 0.05, 0.1]

# Results storage
sensitivity_results = {
    'alpha': {},
    'gamma': {},
    'epsilon_decay': {},
    'epsilon_min': {},
}

print(f"Configuration:")
print(f"  Baseline: α={BEST_ALPHA}, γ={BEST_GAMMA}, ε_decay={BEST_EPSILON_DECAY}")
print(f"  Discretization: {DISCRETIZATION_BINS}×{DISCRETIZATION_BINS}")
print(f"  Episodes: {N_EPISODES}, Seeds: {N_SEEDS}")

## Sensitivity Analysis: Learning Rate (α)

How does learning rate affect convergence and final performance?

In [ ]:
print("\nSensitivity to LEARNING RATE (α)")
print("=" * 60)

for alpha in ALPHA_VALUES:
    print(f"\nTesting α = {alpha}")
    final_successes = []
    convergence_eps = []
    
    for seed in range(N_SEEDS):
        env = create_env("discrete", ENVIRONMENT, seed=seed)
        agent = QLearning(
            alpha=alpha,
            gamma=BEST_GAMMA,
            epsilon_decay=BEST_EPSILON_DECAY,
        )
        
        agent, rewards, successes, steps = train_agent(
            agent=agent,
            env=env,
            n_episodes=N_EPISODES,
            verbose=False,
        )
        
        final_success = np.mean(successes[-50:])
        final_successes.append(final_success)
        
        # Convergence speed
        converged = np.where(np.array(successes) >= 0.5)[0]
        if len(converged) > 0:
            convergence_eps.append(converged[0])
        else:
            convergence_eps.append(N_EPISODES)
    
    sensitivity_results['alpha'][alpha] = {
        'final_success': np.mean(final_successes),
        'final_success_std': np.std(final_successes),
        'convergence': np.mean(convergence_eps),
    }
    
    print(f"  Success: {np.mean(final_successes):.1%} ± {np.std(final_successes):.1%}")
    print(f"  Convergence: {np.mean(convergence_eps):.0f} ± {np.std(convergence_eps):.0f} episodes")

## Sensitivity Analysis: Discount Factor (γ)

How does the discount factor affect long-horizon planning?

In [ ]:
print("\nSensitivity to DISCOUNT FACTOR (γ)")
print("=" * 60)

for gamma in GAMMA_VALUES:
    print(f"\nTesting γ = {gamma}")
    final_successes = []
    convergence_eps = []
    
    for seed in range(N_SEEDS):
        env = create_env("discrete", ENVIRONMENT, seed=seed)
        agent = QLearning(
            alpha=BEST_ALPHA,
            gamma=gamma,
            epsilon_decay=BEST_EPSILON_DECAY,
        )
        
        agent, rewards, successes, steps = train_agent(
            agent=agent,
            env=env,
            n_episodes=N_EPISODES,
            verbose=False,
        )
        
        final_success = np.mean(successes[-50:])
        final_successes.append(final_success)
        
        converged = np.where(np.array(successes) >= 0.5)[0]
        if len(converged) > 0:
            convergence_eps.append(converged[0])
        else:
            convergence_eps.append(N_EPISODES)
    
    sensitivity_results['gamma'][gamma] = {
        'final_success': np.mean(final_successes),
        'final_success_std': np.std(final_successes),
        'convergence': np.mean(convergence_eps),
    }
    
    print(f"  Success: {np.mean(final_successes):.1%} ± {np.std(final_successes):.1%}")
    print(f"  Convergence: {np.mean(convergence_eps):.0f} ± {np.std(convergence_eps):.0f} episodes")

## Sensitivity Analysis: Epsilon Decay (ε_decay)

How does exploration decay schedule affect learning?

In [ ]:
print("\nSensitivity to EPSILON DECAY (ε_decay)")
print("=" * 60)

for eps_decay in EPSILON_DECAY_VALUES:
    print(f"\nTesting ε_decay = {eps_decay}")
    final_successes = []
    convergence_eps = []
    
    for seed in range(N_SEEDS):
        env = create_env("discrete", ENVIRONMENT, seed=seed)
        agent = QLearning(
            alpha=BEST_ALPHA,
            gamma=BEST_GAMMA,
            epsilon_decay=eps_decay,
        )
        
        agent, rewards, successes, steps = train_agent(
            agent=agent,
            env=env,
            n_episodes=N_EPISODES,
            verbose=False,
        )
        
        final_success = np.mean(successes[-50:])
        final_successes.append(final_success)
        
        converged = np.where(np.array(successes) >= 0.5)[0]
        if len(converged) > 0:
            convergence_eps.append(converged[0])
        else:
            convergence_eps.append(N_EPISODES)
    
    sensitivity_results['epsilon_decay'][eps_decay] = {
        'final_success': np.mean(final_successes),
        'final_success_std': np.std(final_successes),
        'convergence': np.mean(convergence_eps),
    }
    
    print(f"  Success: {np.mean(final_successes):.1%} ± {np.std(final_successes):.1%}")
    print(f"  Convergence: {np.mean(convergence_eps):.0f} ± {np.std(convergence_eps):.0f} episodes")

## Sensitivity Analysis: Min Epsilon (ε_min)

How does minimum exploration rate affect final policy?

In [ ]:
print("\nSensitivity to MINIMUM EPSILON (ε_min)")
print("=" * 60)

for eps_min in EPSILON_MIN_VALUES:
    print(f"\nTesting ε_min = {eps_min}")
    final_successes = []
    convergence_eps = []
    
    for seed in range(N_SEEDS):
        env = create_env("discrete", ENVIRONMENT, seed=seed)
        agent = QLearning(
            alpha=BEST_ALPHA,
            gamma=BEST_GAMMA,
            epsilon_decay=BEST_EPSILON_DECAY,
            epsilon_min=eps_min,
        )
        
        agent, rewards, successes, steps = train_agent(
            agent=agent,
            env=env,
            n_episodes=N_EPISODES,
            verbose=False,
        )
        
        final_success = np.mean(successes[-50:])
        final_successes.append(final_success)
        
        converged = np.where(np.array(successes) >= 0.5)[0]
        if len(converged) > 0:
            convergence_eps.append(converged[0])
        else:
            convergence_eps.append(N_EPISODES)
    
    sensitivity_results['epsilon_min'][eps_min] = {
        'final_success': np.mean(final_successes),
        'final_success_std': np.std(final_successes),
        'convergence': np.mean(convergence_eps),
    }
    
    print(f"  Success: {np.mean(final_successes):.1%} ± {np.std(final_successes):.1%}")
    print(f"  Convergence: {np.mean(convergence_eps):.0f} ± {np.std(convergence_eps):.0f} episodes")

## Summary: Parameter Importance Ranking

In [ ]:
# Compute importance scores (range of performance across parameter values)
importance_scores = {}

# Alpha importance
alpha_successes = [sensitivity_results['alpha'][a]['final_success'] for a in ALPHA_VALUES]
importance_scores['Learning Rate (α)'] = max(alpha_successes) - min(alpha_successes)

# Gamma importance  
gamma_successes = [sensitivity_results['gamma'][g]['final_success'] for g in GAMMA_VALUES]
importance_scores['Discount Factor (γ)'] = max(gamma_successes) - min(gamma_successes)

# Epsilon decay importance
eps_decay_successes = [sensitivity_results['epsilon_decay'][e]['final_success'] for e in EPSILON_DECAY_VALUES]
importance_scores['Epsilon Decay (ε_decay)'] = max(eps_decay_successes) - min(eps_decay_successes)

# Epsilon min importance
eps_min_successes = [sensitivity_results['epsilon_min'][e]['final_success'] for e in EPSILON_MIN_VALUES]
importance_scores['Min Epsilon (ε_min)'] = max(eps_min_successes) - min(eps_min_successes)

# Sort by importance
sorted_importance = sorted(importance_scores.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*60)
print("HYPERPARAMETER IMPORTANCE RANKING")
print("(Based on performance range across parameter values)")
print("="*60)
for rank, (param, importance) in enumerate(sorted_importance, 1):
    print(f"{rank}. {param:30s}: {importance:.3f} (Δ Success Rate)")
print("="*60)

In [ ]:
# Update imports to include StatisticalAnalyzer
from src.evaluation import (
    train_agent_legacy as train_agent,
    StatisticalAnalyzer, exponential_smoothing, export_results
)

# Convergence Analysis for Sensitivity Study
analyzer = StatisticalAnalyzer()

print("\n" + "="*70)
print("CONVERGENCE METRICS FOR BEST CONFIGURATION")
print("="*70)

# Extract best results for detailed convergence analysis
best_config = {
    'alpha': BEST_ALPHA,
    'gamma': BEST_GAMMA,
    'epsilon_decay': BEST_EPSILON_DECAY,
    'epsilon_min': BEST_EPSILON_MIN,
}

# Re-train with best config for convergence analysis
best_rewards_across_seeds = []

for seed in range(3):
    env = create_env("discrete", ENVIRONMENT, seed=seed)
    agent = QLearning(
        alpha=BEST_ALPHA,
        gamma=BEST_GAMMA,
        epsilon_decay=BEST_EPSILON_DECAY,
        epsilon_min=BEST_EPSILON_MIN,
    )
    
    agent, rewards, successes, steps = train_agent(
        agent=agent,
        env=env,
        n_episodes=N_EPISODES,
        verbose=False,
    )
    best_rewards_across_seeds.append(rewards)

# Average rewards across seeds
avg_rewards_best = np.mean(best_rewards_across_seeds, axis=0)

# Compute convergence metrics
conv_best = analyzer.compute_convergence_metrics(avg_rewards_best.tolist(), window_size=50)

print(f"\nBest Configuration: α={BEST_ALPHA}, γ={BEST_GAMMA}, ε_decay={BEST_EPSILON_DECAY}, ε_min={BEST_EPSILON_MIN}")
print(f"  Stability Score: {conv_best['stability_score']:.3f}")
print(f"  Improvement (early vs late): {conv_best['improvement']:.2f}")
print(f"  Final Window Mean: {conv_best['final_window_mean']:.2f}")
print(f"  Final Window Std: {conv_best['final_window_std']:.2f}")

print("\n✓ Convergence analysis complete")

## Visualization: Sensitivity Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Alpha sensitivity
ax = axes[0, 0]
alpha_vals = sorted(sensitivity_results['alpha'].keys())
alpha_perf = [sensitivity_results['alpha'][a]['final_success'] for a in alpha_vals]
alpha_std = [sensitivity_results['alpha'][a]['final_success_std'] for a in alpha_vals]
ax.errorbar(alpha_vals, alpha_perf, yerr=alpha_std, fmt='o-', capsize=5, markersize=8, linewidth=2)
ax.axvline(BEST_ALPHA, color='red', linestyle='--', alpha=0.7, label='Current best')
ax.set_xlabel('Learning Rate (α)', fontsize=11)
ax.set_ylabel('Final Success Rate', fontsize=11)
ax.set_title('Sensitivity to Learning Rate', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 1.05)

# Gamma sensitivity
ax = axes[0, 1]
gamma_vals = sorted(sensitivity_results['gamma'].keys())
gamma_perf = [sensitivity_results['gamma'][g]['final_success'] for g in gamma_vals]
gamma_std = [sensitivity_results['gamma'][g]['final_success_std'] for g in gamma_vals]
ax.errorbar(gamma_vals, gamma_perf, fmt='s-', capsize=5, markersize=8, linewidth=2, color='orange')
ax.axvline(BEST_GAMMA, color='red', linestyle='--', alpha=0.7, label='Current best')
ax.set_xlabel('Discount Factor (γ)', fontsize=11)
ax.set_ylabel('Final Success Rate', fontsize=11)
ax.set_title('Sensitivity to Discount Factor', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 1.05)

# Epsilon decay sensitivity
ax = axes[1, 0]
eps_decay_vals = sorted(sensitivity_results['epsilon_decay'].keys())
eps_decay_perf = [sensitivity_results['epsilon_decay'][e]['final_success'] for e in eps_decay_vals]
eps_decay_std = [sensitivity_results['epsilon_decay'][e]['final_success_std'] for e in eps_decay_vals]
ax.errorbar(eps_decay_vals, eps_decay_perf, fmt='^-', capsize=5, markersize=8, linewidth=2, color='green')
ax.axvline(BEST_EPSILON_DECAY, color='red', linestyle='--', alpha=0.7, label='Current best')
ax.set_xlabel('Epsilon Decay (ε_decay)', fontsize=11)
ax.set_ylabel('Final Success Rate', fontsize=11)
ax.set_title('Sensitivity to Exploration Decay', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 1.05)

# Epsilon min sensitivity
ax = axes[1, 1]
eps_min_vals = sorted(sensitivity_results['epsilon_min'].keys())
eps_min_perf = [sensitivity_results['epsilon_min'][e]['final_success'] for e in eps_min_vals]
eps_min_std = [sensitivity_results['epsilon_min'][e]['final_success_std'] for e in eps_min_vals]
ax.errorbar(eps_min_vals, eps_min_perf, fmt='d-', capsize=5, markersize=8, linewidth=2, color='purple')
ax.axvline(BEST_EPSILON_MIN, color='red', linestyle='--', alpha=0.7, label='Current best')
ax.set_xlabel('Min Epsilon (ε_min)', fontsize=11)
ax.set_ylabel('Final Success Rate', fontsize=11)
ax.set_title('Sensitivity to Minimum Exploration', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 1.05)

plt.suptitle('Hyperparameter Sensitivity Analysis', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print("Sensitivity curves plotted.")

## Analysis & Findings

### Key Insights

1. **Most Sensitive Parameter**: The analysis identifies which hyperparameter has the largest impact on performance.

2. **Critical Thresholds**: Some parameters show performance cliffs (e.g., too low γ breaks long-horizon planning).

3. **Robust Parameters**: Some parameters have minimal impact on performance, allowing flexibility in tuning.

4. **Interaction Effects**: While analyzed independently, parameters interact in practice.

### Recommendations

- **If performance degrades**: Check discount factor (γ) first - it has largest impact on long-horizon tasks
- **For speed**: Slightly reduce ε_decay (e.g., 0.99) to speed up learning without major penalty
- **For robustness**: Use ε_min > 0.01 for continued exploration even late in training

### Next Steps

Proceed to **Phase 6: Deep RL** where we scale beyond tabular methods using neural networks (DQN, DDQN).

In [ ]:
print("\n" + "="*60)
print("PHASE 5 SUMMARY")
print("="*60)
print(f"✓ Parameters studied: α, γ, ε_decay, ε_min")
print(f"✓ Sensitivity configurations: {len(ALPHA_VALUES) + len(GAMMA_VALUES) + len(EPSILON_DECAY_VALUES) + len(EPSILON_MIN_VALUES)}")
print(f"✓ Robustness: {N_SEEDS} seeds per configuration")
print(f"✓ Most important parameter: {sorted_importance[0][0]}")
print(f"✓ Performance range: {sorted_importance[0][1]:.3f} (Δ Success Rate)")
print("="*60)